Introduction

In the digital advertising world, predicting ad campaign success is vital for maximizing return on investment. Adbot supports small and medium-sized businesses by offering actionable insights to improve ad campaigns and boost customer engagement.

In digital marketing, clicks refer to when someone views the advert and follows one of the hyperlinks in that advert.

This project aims to predict the number of clicks an ad will receive one and two weeks into the future. Accurate click predictions enable businesses to optimize their campaigns and allocate resources effectively.

The Data

The data provided contains the daily ad records for 185 clients from the Adbot platform

Each record contains information related to the ads including the cost, number of impressions, as well as the number of keywords used in the ads.

The data provided also contains the calls clients receive as a result of hosting ads on the platform. These detail the daily number of received/missed calls as well as the duration of each call.


The following describes the columns present in the data.

* impressions: The number of times an ad is displayed.
* clicks: The number of times an ad is clicked.
* cost: The total cost of the ad campaign.
* conversions: The number of desired actions taken by users (e.g., purchases) after clicking the ad.
* ad_type: The type of ad (e.g., EXPANDED_TEXT_AD).
* currency: The currency in which the cost is measured (e.g., ZAR).
* ID: The unique identifier for the ad campaign.
* date: The date of the ad campaign.
* call_type: The type of call associated with the ad (if applicable).
* call_status: The status of the call associated with the ad (if applicable).
* start_time: The start time of the ad campaign or call.
* duration: The duration of the ad campaign or call.
* end_time: The end time of the ad campaign or call.
* display_location: The location where the ad is displayed.
* impression_share: The share of total impressions the ad receives.
* conversions_calls: The number of conversions from calls (if applicable).
* headline1_len: The length of the first headline in the ad.
* headline2_len: The length of the second headline in the ad.
*ad_description_len: The length of the ad description.

Upload data

In [1]:
from google.colab import files
uploaded = files.upload()

Saving SampleSubmission.csv to SampleSubmission.csv
Saving Train.csv to Train.csv


Importation and loading

To begin, we need to import the libraries that will help us with data manipulation, visualization, and machine learning. We will use pandas for handling data, seaborn for visualizing data, numpy for numerical operations, and scikit-learn for implementing the KNN regression model.

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.neighbors import KNeighborsRegressor

Next, we load our training data and sample submission data from CSV files into pandas DataFrames.

In [3]:
df = pd.read_csv('Train.csv')
submission = pd.read_csv('SampleSubmission.csv')

df.head()

,impressions,clicks,cost,conversions,ad_type,currency,ID,date,call_type,call_status,start_time,duration,end_time,display_location,impression_share,conversions_calls,headline1_len,headline2_len,ad_description_len
0,142.0,15.0,3393.0,0.0,EXPANDED_TEXT_AD,ZAR,ID_5da86e71bf5dee4cf5047046,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,16.279669,0.0,2.0,5.0,11.0
1,89.0,8.0,1817.0,0.0,EXPANDED_TEXT_AD,ZAR,ID_5da86e71bf5dee4cf5047046,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,16.279669,0.0,2.0,2.0,13.0
2,59.0,8.0,1743.0,0.0,EXPANDED_TEXT_AD,ZAR,ID_5da86e71bf5dee4cf5047046,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,16.279669,0.0,2.0,2.0,10.0
3,78.0,4.0,917.0,0.0,EXPANDED_TEXT_AD,ZAR,ID_5da86e71bf5dee4cf5047046,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,16.279669,0.0,2.0,3.0,13.0
4,20.0,1.0,217.0,0.0,EXPANDED_TEXT_AD,ZAR,ID_5da86e71bf5dee4cf5047046,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,16.279669,0.0,2.0,2.0,13.0


In [4]:
submission.head()

,ID,clicks
0,ID_5da86e71bf5dee4cf5047046_2024_01_22,0
1,ID_5da86e71bf5dee4cf5047046_2024_01_29,0
2,ID_5e43c29e6279884e2827d894_2024_02_21,0
3,ID_5e43c29e6279884e2827d894_2024_02_28,0
4,ID_5e4e7b480e374330ee151305_2023_12_04,0


Cleaning the Data

Data cleaning is a fundamental step in any machine learning project. For our project, we will remove any rows with missing values in the clicks column.

In [5]:
df = df.dropna(subset=['clicks']) # delete rows where clicks is NaN
df.reset_index(drop=True, inplace=True) # to make the index continuous

After cleaning the data, we select only the columns that are relevant for training our model, such as ID, date, and clicks. We then sort the data by ID and date to ensure it is organized correctly for further processing.

In [6]:
df = df[['ID', 'date', 'clicks']]
df = df.sort_values(by=['ID', 'date']).reset_index(drop=True)

We were given historical data for each client ID, and each client has different start and end dates. We need to identify the last recorded date for each ID and forecast clicks for the next two weeks. The file ‘SampleSubmission’ specifies the exact dates required.

Let’s consider an example: ID ID_5da86e71bf5dee4cf5047046 has January 14, 2024, as its last recorded date. The dates listed in ‘SampleSubmission’ for this ID are:

- ID_5da86e71bf5dee4cf5047046_2024_01_22

- ID_5da86e71bf5dee4cf5047046_2024_01_29

The dates January 22, 2024, and January 29, 2024, are respectively one and two weeks from January 14, 2024

Therefore, we need to predict the number of clicks for each ID per day. So it makes sense to group by ID and date.

In [7]:
grouped_df = df.groupby(['date', 'ID']).sum().reset_index()
grouped_df['date'] = pd.to_datetime(grouped_df['date']) # set date to datetime

Machine Learning

Defining the KNN Forecast Function
We define a function that uses KNN regression to forecast future values of the clicks time series. This function is central to our project as it enables us to predict future clicks based on past data. The function prepares the training data by creating an array of indices and values. It then initializes and trains the KNN model using this data. After training, the model predicts the future values based on the indices of the values we want to forecast.

In [8]:
forecast_columns = ['clicks']
# This function uses KNN to forecast future values of a time series using the index as input.
def knn_forecast(series, window, forecast_horizon, n_neighbors):
# it takes in a series(the series is the 'clicks' column in this case),
# window size(num of past data in the series to consider - in this case I used 13 (approx 2 weeks in the past)),
# forecast horizon (num of days to forecast into the future - in this case I used 16 (approx 2 weeks into the future)),
# and number of neighbors (number of neighbours to consider in the KNN model - in this case I used 1)
    # Prepare training data for KNN
    X_train = np.arange(len(series)).reshape(-1, 1) # index of the series
    y_train = series.values # values of the series

    # Initialize KNN model
    knn = KNeighborsRegressor(n_neighbors=n_neighbors)
    knn.fit(X_train, y_train) # input is index, output is value

    # Predict the next values using the trained KNN model
    last_index = len(series) - 1 # index of the last value in the series
    X_forecast = np.arange(last_index + 1, last_index + 1 + forecast_horizon).reshape(-1, 1) # index of the values to forecast so as to predict the values for that index
    forecast = knn.predict(X_forecast)

    return forecast # return the forecasted values

Adding Forecasts to Data

Another function is defined to add the forecasted clicks for each specific ID. This function processes each group of data associated with a unique ID, sets the date column as the index, and ensures that the data is in a daily frequency with any missing values forward-filled. It then generates forecast dates and uses the previously defined KNN forecast function to predict future clicks. The result is a DataFrame containing both the original and forecasted data for each ID.

In [9]:
# this fn takes in group (a df with data for one specific ID),
# and returns a lookalike df with forecasted clicks for the next forecast horizon(16days/2weeks)
# based on the past window(13days/2weeks) of clicks from the group
def add_knn_forecasts(group, forecast_horizon, window):
    # group: A DataFrame containing data for one specific ID
    group = group.set_index('date') # set the date column as the index, these are dates for one specific ID
    group.index = pd.to_datetime(group.index) # set the datatype of the index(which is now date) to datetime
    group = group.asfreq('D', method='ffill') # set the frequency of the index to daily and forward fill the missing values

    last_date = group.index.max() # get the last date in the index (last date for that specific ID)
    forecast_dates = pd.date_range(start=last_date, periods=forecast_horizon + 1, freq='D')[1:] # a range of dates to be forecasted (from after the last date to the forecast horizon)
    forecast_data = {} # for storing forecasted values for each column in forecast_columns (in this case, clicks)

    for col in forecast_columns: # in this case, for clicks:
        forecast_data[col] = knn_forecast(group[col], window, forecast_horizon, n_neighbors= window) # clicks = (the fn takes in d clicks column for each date(day) for one specific ID, then returrns the future clicks for the forecast horizon (16days/2weeks))

    forecast_df = pd.DataFrame(forecast_data, index=forecast_dates) # a df containing forcasted clicks with forecast_dates as index
    forecast_df['ID'] = group['ID'].iloc[0] # sets all the ID as the specific ID of this group (since this forecast is for only one ID)
    forecast_df['is_forecast'] = True # indicates thst these rows have been forecasted

    group = group.reset_index() # turning group back to original form
    forecast_df = forecast_df.reset_index().rename(columns={'index': 'date'}) # making forecast_df look exactly like group

    return forecast_df # returns the forecasted df with columns: date, ID, clicks, is_forecast=True)

Applying Forecasts to All Data

We then apply our forecasting function to all groups of ID. By iterating over each group, we generate forecasted data for each unique ID and combine this with the original data. This step is critical as it allows us to create a comprehensive dataset that includes both historical and predicted clicks for all IDs.

In [10]:
all_data = [] # empty list for all data
window_size = 1 # the window_size is the k-nearest_neignbours for the KNN model
forecast_horizon = 16
for name, group in grouped_df.groupby(['ID']): # for each group of specific IDs:
    forecast_df = add_knn_forecasts(group, forecast_horizon, window_size) # get the forecasted df for that specific ID
    all_data.append(pd.concat([group.reset_index(drop=True), forecast_df])) # concat the df for that ID with its forecasted df and append to empty list of all_data

grouped_df = pd.concat(all_data).sort_values(by=['ID', 'date']) # turn the list of all_data to a df and sort by ID and date
grouped_df['is_forecast'] = grouped_df['is_forecast'].fillna(False) # set missing values in is_forecast to False (to show that these rows were not forecasted)

<ipython-input-10-4eebe3f41976>:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  grouped_df['is_forecast'] = grouped_df['is_forecast'].fillna(False) # set missing values in is_forecast to False (to show that these rows were not forecasted)


Splitting Train and Test Data

After generating our forecasts, we split the data into two separate DataFrames: one containing the original data (train) and another containing the forecasted data (test). This separation helps us clearly distinguish between historical and predicted data.

In [11]:
train = grouped_df[grouped_df['is_forecast'] == False]
test = grouped_df[grouped_df['is_forecast'] == True] # test = a df containing only forecasted rows

Preparing the Submission File

Adjusting the Submission DataFrame
Finally, we prepare our submission file. We adjust the sample submission DataFrame to include our forecasted clicks. This involves splitting the ID column to extract the date part and merging it with our test data to ensure all forecasted clicks are accurately reflected in the submission file.

In [12]:
sub = submission.copy()

              # split ID column into date and ID
sub[['year', 'month', 'day']] = sub['ID'].str.extract(r'_(\d{4})_(\d{2})_(\d{2})')
sub['ID'] = sub['ID'].str.replace(r'(_\d{4}_\d{2}_\d{2})$', '', regex=True)
sub['date'] = pd.to_datetime(sub[['year', 'month', 'day']])

In [13]:
filtered_test = pd.merge(test, sub[['ID', 'date']], on=['ID', 'date']) # all rows in test r kept
merge_df = pd.merge(sub, filtered_test, on=['ID', 'date'], how='left') # all rows in sub are kept

In [14]:
merge_df.head()

,ID,clicks_x,year,month,day,date,clicks_y,is_forecast
0,ID_5da86e71bf5dee4cf5047046,0,2024,01,22,2024-01-22,114.0,True
1,ID_5da86e71bf5dee4cf5047046,0,2024,01,29,2024-01-29,114.0,True
2,ID_5e43c29e6279884e2827d894,0,2024,02,21,2024-02-21,6.0,True
3,ID_5e43c29e6279884e2827d894,0,2024,02,28,2024-02-28,6.0,True
4,ID_5e4e7b480e374330ee151305,0,2023,12,04,2023-12-04,2.0,True


In [15]:
click_sums = merge_df.groupby(['ID', 'date'])['clicks_y'].sum().reset_index()
click_sums.rename(columns={'clicks_y': 'sum_clicks'}, inplace=True)

We then round the click values to the nearest whole number and save the final submission file in CSV format.

In [16]:
sub = pd.merge(sub, click_sums, on=['ID', 'date'], how='left')
sub['clicks'] = sub['sum_clicks']

sub.drop(columns='sum_clicks', inplace=True)
sub['clicks'] = sub['clicks'].round() # round to nearest whole number
submission['clicks'] = sub['clicks']

In [17]:
submission.to_csv('submission.csv', index=False)

Conclusion

By following these steps, we have effectively use KNN regression to forecast future clicks for ads, demonstrating a practical application of machine learning in the field of digital marketing. This approach can be further refined and expanded to handle more complex datasets and forecasting requirements.